<a href="https://colab.research.google.com/github/TanujaMore27/Deep-Learning/blob/main/4_pytorch_nn_module_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

In [4]:
class Model(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features,3),
        nn.ReLU(),
        nn.Linear(3,1),
        nn.Sigmoid()
    )

  def forward(self,features):
      out = self.network(features)
      return out

In [5]:
features = torch.rand(10,5)
model = Model(features.shape[1])
#call model for forward pass
model(features)

tensor([[0.5101],
        [0.5025],
        [0.5101],
        [0.5000],
        [0.5101],
        [0.5101],
        [0.5101],
        [0.5101],
        [0.5101],
        [0.5101]], grad_fn=<SigmoidBackward0>)

In [11]:
model.network[0].weight

Parameter containing:
tensor([[ 0.2438, -0.2566,  0.2910, -0.2236, -0.1217],
        [-0.2329, -0.0724, -0.1876,  0.3298, -0.3228],
        [-0.2163, -0.2367,  0.1846,  0.0391, -0.1345]], requires_grad=True)

In [10]:
model.network[2].weight

Parameter containing:
tensor([[-0.3975, -0.4589, -0.0427]], requires_grad=True)

In [14]:
model.network[0].bias

Parameter containing:
tensor([-0.1291, -0.3968, -0.3883], requires_grad=True)

In [13]:
model.network[2].bias

Parameter containing:
tensor([0.0403], requires_grad=True)

In [12]:
!pip install torchinfo

In [15]:
from torchinfo import summary

summary(model,input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Sequential: 1-1                        [10, 1]                   --
│    └─Linear: 2-1                       [10, 3]                   18
│    └─ReLU: 2-2                         [10, 3]                   --
│    └─Linear: 2-3                       [10, 1]                   4
│    └─Sigmoid: 2-4                      [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

In [17]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler

In [18]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [19]:
df.shape

(569, 33)

In [20]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

In [21]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [22]:
x_train,x_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2,random_state=42)

In [23]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [24]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [35]:
x_train_tensor = torch.from_numpy(x_train).float()
x_test_tensor = torch.from_numpy(x_test).float()
y_train_tensor = torch.from_numpy(y_train).float()
y_test_tensor = torch.from_numpy(y_test).float()

In [36]:
x_train_tensor.shape

torch.Size([455, 30])

In [37]:
import torch.nn as nn
class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features,1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):
    out = self.linear(features)
    out = self.sigmoid(out)
    return out


In [38]:
lr = 0.1
epochs = 25

In [39]:
loss_function = nn.BCELoss()

In [42]:
# create model
model = MySimpleNN(x_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(),lr)

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model(x_train_tensor)

  # loss calculate
  loss = loss_function(y_pred, y_train_tensor.view(-1,1))

  # clear gradient
  optimizer.zero_grad()

  # backward pass
  loss.backward()

  # parameters update
  optimizer.step()


  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.7245916128158569
Epoch: 2, Loss: 0.5278630256652832
Epoch: 3, Loss: 0.42757168412208557
Epoch: 4, Loss: 0.36880722641944885
Epoch: 5, Loss: 0.3299562335014343
Epoch: 6, Loss: 0.3021392822265625
Epoch: 7, Loss: 0.2810910642147064
Epoch: 8, Loss: 0.26450538635253906
Epoch: 9, Loss: 0.251024454832077
Epoch: 10, Loss: 0.23979610204696655
Epoch: 11, Loss: 0.23025791347026825
Epoch: 12, Loss: 0.22202369570732117
Epoch: 13, Loss: 0.21481910347938538
Epoch: 14, Loss: 0.20844371616840363
Epoch: 15, Loss: 0.20274758338928223
Epoch: 16, Loss: 0.19761614501476288
Epoch: 17, Loss: 0.1929602473974228
Epoch: 18, Loss: 0.18870937824249268
Epoch: 19, Loss: 0.1848069727420807
Epoch: 20, Loss: 0.1812070608139038
Epoch: 21, Loss: 0.17787183821201324
Epoch: 22, Loss: 0.17476987838745117
Epoch: 23, Loss: 0.17187482118606567
Epoch: 24, Loss: 0.16916437447071075
Epoch: 25, Loss: 0.1666194647550583


In [43]:
with torch.no_grad():
  y_pred = model.forward(x_test_tensor)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')

Accuracy: 0.5646352767944336
